## 1. Setup and Data Loading
Imports libraries and loads the protein dataset.

# CNN + Transfomers


In [2]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from torch.nn.utils.rnn import pad_sequence
from tqdm import tqdm
import torch.nn as nn
import numpy as np
import math

# Load the dataset
df = pd.read_csv('/home/users/ntu/ktang022/scratch/SC4001_Assignment2/data/2018-06-06-ss.cleaned.csv')

# Ensure there's a 'len' column with sequence lengths
if 'len' not in df.columns:
    df['len'] = df['seq'].str.len()

print(df.head())
df.info()


  pdb_id chain_code  seq sst8 sst3  len  has_nonstd_aa
0   1A30          C  EDL  CBC  CEC    3          False
1   1B05          B  KCK  CBC  CEC    3          False
2   1B0H          B  KAK  CBC  CEC    3          False
3   1B1H          B  KFK  CBC  CEC    3          False
4   1B2H          B  KAK  CBC  CEC    3          False
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 393732 entries, 0 to 393731
Data columns (total 7 columns):
 #   Column         Non-Null Count   Dtype 
---  ------         --------------   ----- 
 0   pdb_id         393732 non-null  object
 1   chain_code     393732 non-null  object
 2   seq            393732 non-null  object
 3   sst8           393732 non-null  object
 4   sst3           393732 non-null  object
 5   len            393732 non-null  int64 
 6   has_nonstd_aa  393732 non-null  bool  
dtypes: bool(1), int64(1), object(5)
memory usage: 18.4+ MB


In [3]:
# --- 1. Define Vocabulary ---
# Standard 20 Amino Acids + special tokens
aa_list = "ACDEFGHIKLMNPQRSTVWY"
aa_vocab = {aa: i+1 for i, aa in enumerate(aa_list)} # Start from 1
aa_vocab['<pad>'] = 0
aa_vocab['<unk>'] = 21 # For any non-standard AA
vocab_size = len(aa_vocab)

print(f"Vocab Size: {vocab_size}")

# --- 2. Tokenization Helper Functions ---
def tokenize_seq(seq, vocab, max_len=None):
    # Convert to integers, use <unk> for unknown characters (like 'X', 'B', 'Z')
    tokens = [vocab.get(c, vocab['<unk>']) for c in seq]
    return torch.tensor(tokens, dtype=torch.long)

def encode_labels(ss_labels, vocab):
    encoded = []
    for ss in ss_labels:
        ids = [vocab.get(c, -1) for c in ss] # -1 for unknown/padding labels
        encoded.append(torch.tensor(ids, dtype=torch.long))
    return encoded # We will pad later in the collate_fn or dataset

# Prepare Labels
ss8_vocab = {'H': 0, 'G': 1, 'I': 2, 'E': 3, 'B': 4, 'T': 5, 'S': 6, 'C': 7}
ss3_vocab = {'H': 0, 'E': 1, 'C': 2}

# Pre-tokenize all data to save time during training
print("Tokenizing data...")
all_seq_tokens = [tokenize_seq(s, aa_vocab) for s in df['seq']]
all_ss8_labels = encode_labels(df['sst8'], ss8_vocab)
all_ss3_labels = encode_labels(df['sst3'], ss3_vocab)

# --- 3. Custom Dataset ---
class ProteinSeqDataset(Dataset):
    def __init__(self, seq_tokens, ss8_labels, ss3_labels):
        self.seq_tokens = seq_tokens
        self.ss8_labels = ss8_labels
        self.ss3_labels = ss3_labels

    def __len__(self):
        return len(self.seq_tokens)

    def __getitem__(self, idx):
        return self.seq_tokens[idx], self.ss8_labels[idx], self.ss3_labels[idx]

# --- 4. Collate Function for Padding ---
# This efficiently pads every batch to the longest sequence IN THAT BATCH, rather than the global max.
# drastically speeds up training compared to global padding.
def collate_batch(batch):
    seqs, ss8, ss3 = zip(*batch)
    # Pad sequences with 0 (<pad> token)
    seqs_padded = pad_sequence(seqs, batch_first=True, padding_value=0)
    # Pad labels with -1 (ignore index for loss)
    ss8_padded = pad_sequence(ss8, batch_first=True, padding_value=-1)
    ss3_padded = pad_sequence(ss3, batch_first=True, padding_value=-1)
    return seqs_padded, ss8_padded, ss3_padded

Vocab Size: 22
Tokenizing data...


## 5. Split Data and Create Dataloaders
Same as your original notebook.

In [4]:
# Split indices
indices = np.arange(len(df))
train_idx, temp_idx = train_test_split(indices, test_size=0.2, random_state=42)
val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, random_state=42)

# Create Datasets based on indices
train_ds = ProteinSeqDataset([all_seq_tokens[i] for i in train_idx],
                             [all_ss8_labels[i] for i in train_idx],
                             [all_ss3_labels[i] for i in train_idx])
val_ds = ProteinSeqDataset([all_seq_tokens[i] for i in val_idx],
                           [all_ss8_labels[i] for i in val_idx],
                           [all_ss3_labels[i] for i in val_idx])
test_ds = ProteinSeqDataset([all_seq_tokens[i] for i in test_idx],
                            [all_ss8_labels[i] for i in test_idx],
                            [all_ss3_labels[i] for i in test_idx])

# Create DataLoaders with custom collate function
BATCH_SIZE = 32 # Increased batch size slightly as we don't have huge ESM embeddings in memory
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_batch)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_batch)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_batch)

## 6. Define the CNN + Transfomer Model
**MODIFIED:** This cell defines a 1D CNN model instead of a Transformer. It's designed to accept the exact same input shape `(batch_size, seq_len, embedding_dim)`.

In [5]:
class PositionalEncoding(nn.Module):
    # (Kept same as your original notebook)
    def __init__(self, d_model, dropout=0.1, max_len=6000):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, 1, d_model)
        pe[:, 0, 0::2] = torch.sin(position * div_term)
        pe[:, 0, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:x.size(1), :].transpose(0, 1)
        return self.dropout(x)

class TransformerCNN(nn.Module):
    def __init__(self, vocab_size, d_model=256, nhead=8, num_trans_layers=4,
                 cnn_channels=128, cnn_kernel=5, dropout=0.1):
        super().__init__()

        # 1. Learnable standard embeddings (Integer -> Dense Vector)
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos_encoder = PositionalEncoding(d_model, dropout)

        # 2. Transformer Encoder (Creates contextualized embeddings)
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead,
                                                   dim_feedforward=d_model*4,
                                                   dropout=dropout, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_trans_layers)

        # 3. CNN Layer (Processes transformer output locally)
        # Input to CNN will be [Batch, d_model, SeqLen]
        self.cnn = nn.Conv1d(in_channels=d_model, out_channels=cnn_channels,
                             kernel_size=cnn_kernel, padding=cnn_kernel//2)
        self.relu = nn.ReLU()
        self.dropout_cnn = nn.Dropout(dropout)

        # 4. Classification Heads
        self.q8_head = nn.Linear(cnn_channels, 8)
        self.q3_head = nn.Linear(cnn_channels, 3)

    def forward(self, x, mask_padding=None):
        """
        x: [batch_size, seq_len] (Integer tokens)
        mask_padding: [batch_size, seq_len] (True where padding is 0)
        """
        # A. Create Embeddings via Transformer
        x = self.embedding(x) # -> [B, L, d_model]
        x = self.pos_encoder(x)

        # Transformer requires a boolean mask where True = ignore position
        # We use the `mask_padding` generated in the training loop for this.
        context_embeddings = self.transformer(x, src_key_padding_mask=mask_padding) # -> [B, L, d_model]

        # B. Then do it via CNN
        # Permute for CNN: [B, L, D] -> [B, D, L]
        cnn_in = context_embeddings.permute(0, 2, 1)
        cnn_out = self.relu(self.cnn(cnn_in))
        cnn_out = self.dropout_cnn(cnn_out)

        # Permute back for linear heads: [B, CNN_C, L] -> [B, L, CNN_C]
        out = cnn_out.permute(0, 2, 1)

        return self.q8_head(out), self.q3_head(out)

## 7. Training Loop
**MODIFIED:** This now instantiates `ProteinCNN` instead of `ProteinTransformer`. The rest of the logic is identical, as the inputs and outputs are the same.

In [ ]:
# Initialize Model
# d_model=256, 4 transformer layers is a decent starting point for training from scratch
model = TransformerCNN(vocab_size=vocab_size, d_model=256, nhead=8, num_trans_layers=4,
                       cnn_channels=128, cnn_kernel=5, dropout=0.2)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Optimizer & Loss
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4) # AdamW usually better for Transformers
criterion_q8 = nn.CrossEntropyLoss(ignore_index=-1)
criterion_q3 = nn.CrossEntropyLoss(ignore_index=-1)

def compute_accuracy(pred_logits, labels):
    preds = pred_logits.argmax(-1)
    mask = labels != -1
    correct = (preds[mask] == labels[mask]).sum().item()
    total = mask.sum().item()
    return correct / total if total > 0 else 0.0

# Training Setup
num_epochs = 100
best_val_acc = 0.0
patience = 8 # Slightly higher patience when training from scratch
patience_counter = 0
model_s_path= "best_trans_cnn_model.pt"
print("Starting training...")
for epoch in range(num_epochs):
    model.train()
    train_loss_avg, train_q8_acc, train_q3_acc = 0, 0, 0

    for seqs, ss8, ss3 in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}"):
        seqs, ss8, ss3 = seqs.to(device), ss8.to(device), ss3.to(device)

        # Create padding mask for Transformer (True where input is <pad> (0))
        padding_mask = (seqs == 0)

        # Forward pass
        q8_logits, q3_logits = model(seqs, mask_padding=padding_mask)

        # Flatten for loss
        loss_q8 = criterion_q8(q8_logits.reshape(-1, 8), ss8.reshape(-1))
        loss_q3 = criterion_q3(q3_logits.reshape(-1, 3), ss3.reshape(-1))
        loss = loss_q8 + 0.5 * loss_q3

        optimizer.zero_grad()
        loss.backward()
        # Optional: Gradient clipping to prevent exploding gradients in Transformers
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        train_loss_avg += loss.item()
        train_q8_acc += compute_accuracy(q8_logits, ss8)
        train_q3_acc += compute_accuracy(q3_logits, ss3)

    # Validation
    model.eval()
    val_loss_avg, val_q8_acc, val_q3_acc = 0, 0, 0
    with torch.no_grad():
        for seqs, ss8, ss3 in val_loader:
            seqs, ss8, ss3 = seqs.to(device), ss8.to(device), ss3.to(device)
            padding_mask = (seqs == 0)
            q8_logits, q3_logits = model(seqs, mask_padding=padding_mask)
            
            loss_q8 = criterion_q8(q8_logits.reshape(-1, 8), ss8.reshape(-1))
            loss_q3 = criterion_q3(q3_logits.reshape(-1, 3), ss3.reshape(-1))
            val_loss_avg += (loss_q8 + 0.5 * loss_q3).item()
            val_q8_acc += compute_accuracy(q8_logits, ss8)
            val_q3_acc += compute_accuracy(q3_logits, ss3)

    # Logging
    train_loss_avg /= len(train_loader)
    val_loss_avg /= len(val_loader)
    val_q8_acc /= len(val_loader)
    
    print(f"Epoch {epoch+1}: Val Loss={val_loss_avg:.4f}, Val Q8 Acc={val_q8_acc:.4f}")

    # Early Stopping
    if val_q8_acc > best_val_acc:
        best_val_acc = val_q8_acc
        torch.save(model.state_dict(), model_s_path)
        patience_counter = 0
        print("Best model saved!")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print("Early stopping.")
            break

Starting training...


Epoch 1/100: 100%|██████████| 9844/9844 [15:45<00:00, 10.41it/s]
/home/users/ntu/ktang022/.conda/envs/myvenv/lib/python3.10/site-packages/torch/nn/modules/transformer.py:515: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


Epoch 1: Val Loss=1.4663, Val Q8 Acc=0.5876
Best model saved!


Epoch 2/100: 100%|██████████| 9844/9844 [15:43<00:00, 10.43it/s]


Epoch 2: Val Loss=1.3041, Val Q8 Acc=0.6368
Best model saved!


...

Epoch 13/100: 100%|██████████| 9844/9844 [15:44<00:00, 10.42it/s]


Epoch 13: Val Loss=0.9744, Val Q8 Acc=0.7302
Best model saved!


Epoch 14/100: 100%|██████████| 9844/9844 [15:42<00:00, 10.44it/s]


Epoch 14: Val Loss=0.9797, Val Q8 Acc=0.7323
Best model saved!


Epoch 15/100:   3%|▎         | 253/9844 [00:22<15:15, 10.48it/s]

## 8. Final Evaluation on Test Set
**MODIFIED:** Loads the saved `best_cnn_model.pt`.

In [ ]:
print("\nLoading best model for testing...")
# Re-init model to ensure clean state (optional but good practice)
best_model = TransformerCNN(VOCAB_SIZE, D_MODEL, NHEAD, NUM_TRANS_LAYERS,
                            CNN_CHANNELS, CNN_KERNEL, DROPOUT)
best_model.load_state_dict(torch.load(model_s_path))
best_model.to(device)
best_model.eval()

test_loss, test_q8, test_q3 = 0, 0, 0
with torch.no_grad():
    for seqs, ss8, ss3 in tqdm(test_loader, desc="Testing"):
        seqs, ss8, ss3 = seqs.to(device), ss8.to(device), ss3.to(device)
        padding_mask = (seqs == 0)
        q8_logits, q3_logits = best_model(seqs, padding_mask)
        
        test_loss += (criterion_q8(q8_logits.reshape(-1, 8), ss8.reshape(-1)) + 
                      0.5 * criterion_q3(q3_logits.reshape(-1, 3), ss3.reshape(-1))).item()
        test_q8 += compute_accuracy(q8_logits, ss8)
        test_q3 += compute_accuracy(q3_logits, ss3)

print(f"\n=== Final Test Results ===")
print(f"Test Loss: {test_loss / len(test_loader):.4f}")
print(f"Test Q8 Accuracy: {test_q8 / len(test_loader):.4f}")
print(f"Test Q3 Accuracy: {test_q3 / len(test_loader):.4f}")